# Banc d'essai `bench` — génération AP-HP

Notebook du répertoire `generation/` (spec : `docs/spec_testrun_run_stage.md`),
suite courte d'appels au package `bench` : les fonctions de données sont dans
`bench/scenarios.py`, l'orchestration (gardes, idempotence, messages) dans
`bench/banc.py`. Cycle de vie d'un test : environnement → pool candidat →
montage → seeding/figement → génération → bilan. Toutes les étapes qui
touchent le disque sont **idempotentes** : elles s'exécutent si nécessaire,
sinon elles skippent avec un message d'état — le notebook se ré-exécute
intégralement sur un test déjà préparé. Rien ne se nettoie : pour repartir
de zéro, on crée `runs/NN+1`. Le disque fait foi ; **tout `generation/` est
versionné et partagé** (§9) — « figer » un jeu = commiter son test.

**Mode d'emploi.** Éditer la cellule **PARAMÈTRES COURANTS** (test courant,
parquet source, quotas, graine, `ONLY`), puis exécuter tout. Deux gestes à ne
pas confondre : le **montage** installe le jeu de templates du test dans
`system/one_gen/` depuis le test précédent (l'objet versionné) ; le
**seeding puis figement** crée les dossiers CRH depuis les scénarios, puis
résout le jeu par famille dans chaque dossier (la production du run).

- **Essai rapide d'un nouveau parquet** : `QUOTAS = "couverture"` (un
  scénario par type d'hospitalisation présent) et, au run réel,
  `ONLY = ["0000"]` (un seul appel payant). `preparer_pool` commence par le
  contrôle des prérequis du fichier (`verifier_source`) : colonnes, types,
  encodages, types d'hospitalisation affichés.
- **Itérer après édition du jeu** : changer `SYSTEM_PROMPT_FILE` et
  `OUT_FILE` (paramètres avancés) — les variantes coexistent, rien n'est
  écrasé. Le jeu s'édite dans `runs/<TEST_NUM>/system/one_gen/`.
- **Clé API** : exclusivement `MISTRAL_API_KEY` dans l'environnement du
  noyau, jamais dans le notebook. Tout s'exécute **sans clé** jusqu'aux
  dry-runs inclus ; seules les cellules « run réel » l'exigent.
- Annexes (2-gen, `prompt_local.py`, itération par copie, ajout de DAS) :
  `notebook_annexes.ipynb`.

In [ ]:
# --- PARAMÈTRES COURANTS — la seule cellule éditée au quotidien ---
TEST_NUM = "06"    # le test courant (étiquettes tabac/alcool = données, bloc H)
PREV_TEST = "05"   # test précédent de la chaîne (None pour un tout premier test)

# Le parquet de scénarios (chemin depuis la racine du repo, ou absolu) — à
# garder IDENTIQUE d'un test à l'autre de la chaîne (comparabilité).
SOURCE_PROFILES_PATH = "data/aphp/scenarios_bn_all_20260128.pq"

# Tirage : dict {modalité de QUOTAS_BY: effectif} (un quota à 0 documente une
# strate volontairement exclue ; la somme = la taille du test), OU
# "couverture" (1 scénario par type présent dans le fichier — le smoke d'un
# nouveau parquet), OU None (tirage simple de TARGET_N séjours).
QUOTAS = {
     "Séances simples": 2,
     "HDJ médecine adultes": 2,
     "Médecine adultes > 3 nuits": 2,
     "Chirurgie adultes < 3 nuits": 2,
     "Chirurgie adultes > 3 nuits": 2,
     "Interventionnel adultes < 3 nuits":1,
     "Interventionnel adultes > 3 nuits":1,
     "Accouchement normal mère": 1,
     "Bébé normal": 0,
     "IMG & fausses couches":1,
     "IVG": 0,  # DP en 8 : aucune IVG disponible
     "Greffes de moelle, CAR-T Cells": 0,
     "Brûlés" : 0,
     "Transplantations" : 0,
}
RANDOM_SEED = 42

ONLY = None  # ex. ["0000", "0001"] : run réel partiel (entrée `partial` au journal)

In [ ]:
# --- Bootstrap (ne pas toucher) : racine du repo, import de bench ---
import os
import sys
from pathlib import Path

REPO_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                  if (p / "bench").is_dir() and (p / "core").is_dir()), None)
assert REPO_ROOT, "Racine du repo Stream introuvable — lancer le notebook depuis generation/."
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import polars as pl
from bench import BenchError, Pricing, generate, load_reports, scenario_dirs
from bench.banc import *
from enrichissement import Politique

# --- PARAMÈTRES AVANCÉS (rarement touchés) ---
MODEL = os.environ.get("MISTRAL_MODEL", "mistral-large-latest")
MAX_TOKENS_SUMMARY = 8_000
MAX_TOKENS_CR = 128_000
# Transport des appels Mistral (spec v3.7) : "sync" = un chat.complete par
# scénario, MAX_WORKERS en parallèle ; "batch" = 50 % moins cher mais reste en
# file indéfiniment depuis septembre 2026 — à réactiver quand Mistral l'aura
# rétabli. Les tarifs (USD / 1M tokens, Mistral Large, https://mistral.ai/pricing)
# suivent le transport.
TRANSPORT = os.environ.get("MISTRAL_TRANSPORT", "sync")
MAX_WORKERS = 3
_TARIFS = {"sync": (0.5, 1.5), "batch": (0.25, 0.75)}
PRICING = Pricing(*_TARIFS[TRANSPORT])

# Sélection et enrichissement — à garder IDENTIQUES d'un test à l'autre de
# la chaîne (comparabilité).
QUOTAS_BY = "DPEC"          # ou "TPEC" — clés de QUOTAS
FILTRE_DP_SUFFIXE = "8"     # séjours dont le DP se termine par « 8 » (sous-
                            # catégories « autres formes précisées ») ; None = pas de filtre
TARGET_N = 15               # tirage simple, quand QUOTAS = None
SCENARIO_FILTERS: list[dict] = [
    # ex. {"column": "template_name", "op": "eq", "value": "surgery_outpatient.txt"},
]
RANDOM_SELECTION = True
ENRICHIR_SCENARIOS = True   # lot E1 : DAS tabac/alcool/corpulence + contexte patient
ENRICHISSEMENT_SEED = RANDOM_SEED
POLITIQUE_ENRICHISSEMENT = Politique()

# Fichiers du run — pour ITÉRER après édition du jeu : changer
# SYSTEM_PROMPT_FILE (ex. prompt_system_one_gen_v2.txt) et OUT_FILE (ex.
# crh_v2.txt) — les variantes coexistent, rien n'est écrasé.
SYSTEM_PROMPT_FILE = "prompt_system_one_gen.txt"
OUT_FILE = "crh_generation.txt"
PARAMS_GEN = dict(
    system=SYSTEM_PROMPT_FILE, user="user_generation.txt", out=OUT_FILE,
    model=MODEL, max_tokens=MAX_TOKENS_CR, pricing=PRICING, prefix_file="prefix.txt",
)

# Vérificateur (optionnel) : textes d'exemple, à adapter à la campagne.
VERIF_SYSTEM = """Tu es un médecin DIM. On te fournit un compte rendu
hospitalier généré automatiquement. Vérifie sa cohérence clinique et sa
conformité aux règles de codage, puis rends un verdict structuré :
CONFORME ou NON CONFORME, suivi de la liste des anomalies constatées."""
VERIF_USER = "Vérifie le compte rendu suivant et rends ton verdict."
VERIF_HEADER = "### COMPTE RENDU À VÉRIFIER"
VERIF_FOOTER = "### FIN DU COMPTE RENDU"
PARAMS_VERIF = dict(
    system="prompt_system_verif.txt", user="user_verification.txt", out="verdict.txt",
    model=MODEL, max_tokens=MAX_TOKENS_SUMMARY, pricing=PRICING,
    context_header=VERIF_HEADER, context_footer=VERIF_FOOTER,
)

pool = None  # rempli par preparer_pool (section 2)
print("Modèle :", MODEL)
print("Transport :", TRANSPORT, "— tarifs ($ / 1M tokens) :", PRICING)
print("MISTRAL_API_KEY présente :", "MISTRAL_API_KEY" in os.environ)

## 1. Environnement et pool candidat

`verifier_environnement()` : fictomed est l'éditable attendu (clone
`~/Documents/fictomed`, branche `prompt-work` — `uv sync` / `uv run`
réinstallent le PyPI 0.1.2 cassé : refaire `uv pip install -e` puis
redémarrer le noyau), les bibliothèques de fiches respectent le contrat
recode-icd (`bench.fiches`, l'index fait foi) et le lecteur fictomed les lit.

`preparer_pool(...)` : contrôle des prérequis du parquet (`verifier_source`,
schéma `SCHEMA_SOURCE`), typologie TPEC/DPEC (conservée si le fichier la
fournit, sinon calculée), identifiants de traçabilité, filtre DP, tirage
(`QUOTAS`), enrichissement, contrôle du contrat côté pool (codes sans fiche
journalisés, codes ajoutés émissibles), couverture par type. Le pool sort
**prêt pour fictomed** — aucun appel Mistral ici.

In [ ]:
verifier_environnement()

In [ ]:
pool = preparer_pool(
    SOURCE_PROFILES_PATH, QUOTAS, RANDOM_SEED,
    by=QUOTAS_BY, enrichir=ENRICHIR_SCENARIOS,
    enrichissement_seed=ENRICHISSEMENT_SEED, politique=POLITIQUE_ENRICHISSEMENT,
    filtre_dp_suffixe=FILTRE_DP_SUFFIXE, target_n=TARGET_N,
)

## 2. Test courant — montage, seeding puis figement

**Le jeu du test s'édite LÀ : `runs/<TEST_NUM>/system/one_gen/` — un `.txt`
par famille clinique.** Il est monté par copie du jeu du test précédent (la
chaîne des tests est la chaîne des versions) ; pour un tout premier test,
fournir un jeu initial à la main (les jeux historiques sont dans git :
`generation/runs/01`). Un `prefix.txt` dans le jeu remplace le prefix
fictomed au seeding ; quand `ENRICHIR_SCENARIOS` est actif, le contexte
patient (taille, poids, IMC, tabac, alcool) est fourni dans le user prompt et
le prompt système doit le restituer (bloc H du jeu de `runs/05`).

`seeder` : génération fictomed (un scénario par ligne du pool ; skip si le
test est déjà seedé), graine (dossiers CRH, `test.json` annoté), puis
figement du jeu par famille en `SYSTEM_PROMPT_FILE` (skip si déjà figé).

In [ ]:
TD, PREV_TD = dossiers_test(TEST_NUM, PREV_TEST)
etat_test(TD, PREV_TD)
monter_jeu(TD, PREV_TD)

In [ ]:
selected_scenarios = seeder(
    TD, pool, source_path=SOURCE_PROFILES_PATH,
    enrichir=ENRICHIR_SCENARIOS, enrichissement_seed=ENRICHISSEMENT_SEED,
    politique=POLITIQUE_ENRICHISSEMENT, seed=RANDOM_SEED,
    scenario_filters=SCENARIO_FILTERS, random_selection=RANDOM_SELECTION,
    system_prompt_file=SYSTEM_PROMPT_FILE,
)

## 3. Génération — contrôle à sec puis run réel

Le contrôle à sec ne fait aucun appel API et aucune écriture — pas besoin de
clé. C'est aussi le **test de complétude** : `generate` échoue (`BenchError`)
au moindre fichier manquant, aucun dossier n'est sauté en silence. Le run
réel écrase `OUT_FILE` à chaque re-run (geste normal) ; chaque run réel
ajoute son entrée au journal `usage.json` (et une ligne par scénario au
journal CSV global `generation/usage_log.csv`). `ONLY` restreint le run réel
à quelques dossiers (entrée `partial` au journal).

In [ ]:
dry = generate(TD, client=None, dry_run=True, **PARAMS_GEN)  # inutile de fournir un client
show_first_prompt(dry)

In [ ]:
client = mistral_client()  # échoue ici, clairement, si MISTRAL_API_KEY absente

cr = generate(
    TD, client=client, transport=TRANSPORT, max_workers=MAX_WORKERS,
    only=ONLY, **PARAMS_GEN,
)
print(cr.usage)

In [ ]:
# Lecture des CR : rendu markdown inline du premier scénario, puis un aperçu
# .md à côté de chaque .txt du test (ex. 0007/crh_generation.md, versionné —
# « Markdown: Open Preview », ⇧⌘V). afficher_crh(TD, "0007") pour un autre.
_names = scenario_dirs(TD)
if _names:
    afficher_crh(TD, _names[0], OUT_FILE)
else:
    print("Pas encore de dossiers scénario dans", TD)
ecrire_apercus_md(TD, OUT_FILE)

## 4. Vérificateur (optionnel) et bilan

Verdict nourri par les CR de la section 3 — depuis le disque (`load_reports`,
le disque fait foi ; `strict=False` charge les scénarios déjà servis et
`only=` restreint le run à ceux-là). Les prompts partagés du vérificateur sont
posés une fois (`write_prompts`) ; le contrôle à sec accepte un contexte
placeholder sans run réel.

Bilan : journal **append-only** des coûts (`committed_usd` = total engagé,
`current_usd` = coût de l'état courant), stats du journal CSV global, puis
les contrôles mécaniques hors modèle (`scripts/check_crh.py`,
`scripts/reancre_crh.py`, stdlib uniquement).

In [ ]:
prompts_verificateur(TD, VERIF_SYSTEM, VERIF_USER)
ctx = contexte_verificateur(TD, OUT_FILE)
dry_verif = generate(TD, client=None, dry_run=True, context=ctx, **PARAMS_VERIF)
show_first_prompt(dry_verif)

In [ ]:
client = mistral_client()

cr_disque = load_reports(TD, OUT_FILE, strict=False)
if cr_disque.height == 0:
    raise RuntimeError(f"Aucun {OUT_FILE} sur disque : lancer d'abord le run réel (section 3).")

verdicts = generate(
    TD, client=client, transport=TRANSPORT, max_workers=MAX_WORKERS,
    context=cr_disque, only=cr_disque["scenario"].to_list(), **PARAMS_VERIF,
)
print(verdicts.usage)

In [ ]:
bilan(TD, out_file=OUT_FILE)